In [2]:
!pip install pandas nltk


In [4]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download stopwords
nltk.download('stopwords')

# Load datasets
df_fake = pd.read_csv(r"C:/Users/hp/MLPROJECT/dataset/Fake.csv")
df_true = pd.read_csv(r"C:/Users/hp/MLPROJECT/dataset/True.csv")

# Add labels
df_fake["label"] = 0  # Fake News
df_true["label"] = 1  # True News

# Merge datasets
df = pd.concat([df_fake, df_true], axis=0).reset_index(drop=True)

# Display first few rows
df.head()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [5]:
# Initialize Stopwords & Stemmer
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def clean_text(text):
    text = str(text).lower()  # Convert to lowercase
    text = re.sub(r'\W+', ' ', text)  # Remove special characters
    text = re.sub(r'\d+', '', text)  # Remove numbers
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]  # Stemming & Stopword removal
    return " ".join(words)

# Apply cleaning function to text column
df["cleaned_text"] = df["text"].apply(clean_text)

# Display cleaned data
df.head()


,title,text,subject,date,label,cleaned_text
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0,donald trump wish american happi new year leav...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0,hous intellig committe chairman devin nune go ...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0,friday reveal former milwauke sheriff david cl...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0,christma day donald trump announc would back w...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0,pope franci use annual christma day messag reb...


In [7]:
df.to_csv("cleaned_news.csv", index=False)
print("✅ Cleaned dataset saved as 'cleaned_news.csv'")


✅ Cleaned dataset saved as 'cleaned_news.csv'


In [11]:
save_path = r"C:\Users\hp\Documents\ML\cleaned_news.csv"
df.to_csv(save_path, index=False)
print(f"✅ Cleaned dataset saved at: {save_path}")


✅ Cleaned dataset saved at: C:\Users\hp\Documents\ML\cleaned_news.csv


In [37]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import os

# ✅ Step 1: Load the Cleaned Dataset
cleaned_file_path = r"C:\Users\hp\Documents\Ml\cleaned_news.csv"  # Ensure this file exists
df = pd.read_csv(cleaned_file_path)

# ✅ Step 2: Check Available Columns
print("Columns in dataset:", df.columns)

# ✅ Step 3: Ensure Required Column Exists
if 'text' not in df.columns:
    raise KeyError("Column 'text' not found! Check the dataset for the correct column name.")

# ✅ Step 4: Normalize Text Length
df["text_length"] = df["text"].apply(lambda x: len(str(x).split()))  # Count words

# ✅ Step 5: Apply Min-Max Scaling
scaler = MinMaxScaler()
df["normalized_length"] = scaler.fit_transform(df[["text_length"]])  # Min-Max Scaling

# ✅ Step 6: Apply TF-IDF Normalization
vectorizer = TfidfVectorizer(max_features=5000)  # Limit features for efficiency
tfidf_matrix = vectorizer.fit_transform(df["text"])

# Convert TF-IDF matrix to DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

# ✅ Step 7: Check Class Distribution (Class Imbalance)
if 'label' in df.columns:
    class_counts = df["label"].value_counts()
    print("Class Distribution:\n", class_counts)
    # Suggest Handling if Imbalance Detected
    imbalance_ratio = class_counts.min() / class_counts.max()
    if imbalance_ratio < 0.5:
        print("⚠ Warning: Dataset has class imbalance! Consider resampling techniques.")

# ✅ Step 8: Save the Normalized Dataset
output_file_path = r"C:\Users\hp\Documents\Ml\normalized_news.csv"
df.to_csv(output_file_path, index=False)
print(f"✅ Normalized dataset saved at: {output_file_path}")


Columns in dataset: Index(['title', 'text', 'subject', 'date', 'label'], dtype='object')
Class Distribution:
 label
0    23481
1    21417
Name: count, dtype: int64
✅ Normalized dataset saved at: C:\Users\hp\Documents\Ml\normalized_news.csv
